In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import tri as mtri
import time

np.random.seed(1234)
torch.manual_seed(1234)
torch.set_default_dtype(torch.float64)
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")  
print("Using device:", device)


In [ ]:
#Material properties
E   = 210e3
nu  = 0.3
Gc  = 2.7

L = 1.0
W = 1.0
xmin, xmax = 0.0, L
ymin, ymax = 0.0, W
ls = 0.0075
plane_strain = True
use_split = True    #Indicator to apply spectral decomposition of the energy 

lam = E*nu / ((1+nu)*(1-2*nu)) if plane_strain else E*nu / (1-nu**2)
mu  = E / (2*(1+nu))

#Incremental loading
nSteps = 18
deltaV = 5.e-4
v_deltas = (np.arange(0, nSteps, dtype=np.float64) * deltaV)


# Training parameters
n_epochs_step0 = 15000
n_epochs_other = 2000
lbfgs_step0_iter = 6000
lbfgs_other_iter = 1000
print_every = 1000
plot_every = 1


In [ ]:
#Grid generation for SENT specimen; fine strip at the crack and notch area
#Note that this block should be changed depending on the example

def gen_grid(L, W, ls):
    inner_offset = 3.0 * ls   # half-width of the fine strip
    outer_offset = 6.0 * ls   # half-width of the transition strip

    sec_bounds = np.array([
        [0.0,                  0.5*W - outer_offset],   
        [0.5*W - outer_offset, 0.5*W - inner_offset],  
        [0.5*W - inner_offset, 0.5*W + inner_offset],   
        [0.5*W + inner_offset, 0.5*W + outer_offset],   
        [0.5*W + outer_offset, W]])                     

    nPred = np.array([[50,  20],    
                      [70, 20],   
                      [140, 25],     
                      [70, 20],    
                      [50,  20]]) 
    
    XY_list, w_list = [], []

    for i in range(5):
        nx, ny = nPred[i]
        y1, y2 = sec_bounds[i]

        xs = np.linspace(0.0, L, nx)
        ys = np.linspace(y1, y2, ny, endpoint=(i == 4))

        X, Y = np.meshgrid(xs, ys)
        XY = np.hstack([X.reshape(-1,1), Y.reshape(-1,1)])
        XY_list.append(XY)

        dx = L / (nx - 1)
        dy = (y2 - y1) / (ny - 1) if ny > 1 else (y2 - y1)
        w = np.full((XY.shape[0],), dx*dy, dtype=np.float64)
        w_list.append(w)

    Grid = np.vstack(XY_list).astype(np.float64)
    w_f  = np.hstack(w_list).astype(np.float64)

    return Grid, w_f

In [ ]:
X_f, w_f = gen_grid(L, W, ls)

#Plot the refined grid
plt.figure(figsize=(6,6))
plt.scatter(X_f[:,0], X_f[:,1], s=1)
plt.title("Piecewise Refined Grid")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()


# spacings at key locations
print(f"Total points: {X_f.shape[0]}")
print(f"Total area (should be 1.0): {w_f.sum():.4f}")
ys_strip = np.unique(X_f[(X_f[:,1] > 0.45) & (X_f[:,1] < 0.55), 1])
i_mid = np.argmin(np.abs(ys_strip - 0.5))
print(f"Δy at y=0.5:       {ys_strip[i_mid+1] - ys_strip[i_mid]:.5f}")
print(f"Δy at strip edge:  {ys_strip[1] - ys_strip[0]:.5f}")
print(f"Δx in strip:       {1.0/299:.5f}")

In [ ]:
B = 1000.0

X_f, w_f = gen_grid(L, W, ls)
Nq = X_f.shape[0]
xg_t = torch.from_numpy(X_f).to(device=device, dtype=torch.get_default_dtype())
wg_t = torch.from_numpy(w_f).to(device=device, dtype=torch.get_default_dtype())

#Rebuild H_init for the new grid
x = X_f[:,0].astype(np.float64)
y = X_f[:,1].astype(np.float64)

x_tip, y_tip = 0.5, 0.5
crackTip = x_tip
dist = np.zeros((Nq,1), dtype=np.float64)
mask_ahead = x > crackTip
mask_behind = ~mask_ahead
dist[mask_behind, 0] = np.abs(y[mask_behind] - y_tip)
dist[mask_ahead,  0] = np.sqrt((x[mask_ahead]-x_tip)**2 + (y[mask_ahead]-y_tip)**2)
inside = dist[:,0] < 0.5*ls

H_init_np = np.zeros((Nq,1), dtype=np.float64)
H_init_np[inside, 0] = B*Gc*0.5*(1.0 - 2.0*dist[inside,0]/ls)/ls
H_init = torch.from_numpy(H_init_np).to(device=device, dtype=torch.get_default_dtype())

#Initialize H from H_init
H = torch.maximum(torch.zeros_like(H_init), H_init.clone())

phi0 = np.zeros((Nq,1), dtype=np.float64)
phi0_t = torch.from_numpy(phi0).to(device=device, dtype=torch.get_default_dtype())

In [ ]:
def strain_calc(u, x):

    ux, uy = u[:, 0:1], u[:, 1:2]

    grad_ux = torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    grad_uy = torch.autograd.grad(uy, x, torch.ones_like(uy), create_graph=True)[0]

    ux_x = grad_ux[:, 0:1]
    ux_y = grad_ux[:, 1:2]
    uy_x = grad_uy[:, 0:1]
    uy_y = grad_uy[:, 1:2]

    eps_xx = ux_x
    eps_yy = uy_y
    eps_xy = 0.5*(ux_y + uy_x)

    return eps_xx, eps_yy, eps_xy

def grad_phi(phi, x):
    gphi = torch.autograd.grad(phi, x, torch.ones_like(phi), create_graph=True)[0]
    return gphi[:,0:1], gphi[:,1:2]

def princ_strains_2d(eps_xx, eps_yy, eps_xy, eps=1e-12):
    tr = eps_xx + eps_yy
    diff = eps_xx - eps_yy
    disc = torch.sqrt(diff**2 + 4.0*eps_xy**2 + 1e-16) 
    e1 = 0.5*(tr + disc)
    e2 = 0.5*(tr - disc)
    return e1, e2

def el_energy(eps_xx, eps_yy, eps_xy, lam, mu):
    tr = eps_xx + eps_yy
    return 0.5*lam*tr**2 + mu*(eps_xx**2 + eps_yy**2 + 2*eps_xy**2)

def el_energy_spectral_split(eps_xx, eps_yy, eps_xy, lam, mu):
    
    e1, e2 = princ_strains_2d(eps_xx, eps_yy, eps_xy)
    tr = e1 + e2

    trp = torch.relu(tr)
    trm = torch.relu(-tr)

    e1p, e2p = torch.relu(e1),  torch.relu(e2)
    e1m, e2m = torch.relu(-e1), torch.relu(-e2)

    Psi_pos = 0.5*lam*trp**2 + mu*(e1p**2 + e2p**2)
    Psi_neg = 0.5*lam*trm**2 + mu*(e1m**2 + e2m**2)

    return Psi_pos, Psi_neg

In [ ]:
def build_mlp(layers, activation=nn.Tanh):
    modules = []
    for i in range(len(layers)-1):
        modules.append(nn.Linear(layers[i], layers[i+1]))
        if i < len(layers)-2:
            modules.append(activation())
    net = nn.Sequential(*modules)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)
    return net

class PINN2D_PF(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.net = build_mlp(layers)

    def forward_raw(self, x):
        lb = torch.tensor([xmin, ymin], device=x.device, dtype=x.dtype)
        ub = torch.tensor([xmax, ymax], device=x.device, dtype=x.dtype)
        x_n = 2.0*(x - lb)/(ub - lb) - 1.0
        return self.net(x_n)


    def apply_BC2(self, x, v_delta, out_raw):

       """
       Hard BCs for SENT — Ghaffari Motlagh et al. (2022).

       Centered coordinates:  x_c = x - 0.5,  y_c = y - 0.5 

       ux = (x_c^2 - (Lx/2)^2) * uh
            → ux = 0  at left  (x = xmin) and right (x = xmax)

        uy = (y_c^2 - (Ly/2)^2) * uh  +  (y_c + Ly/2) * Δv
            → uy = 0     at bottom (y = ymin)
            → uy = Δv    at top    (y = ymax)
       """

       ux_hat  = out_raw[:, 0:1]
       uy_hat  = out_raw[:, 1:2]
       phi_hat = out_raw[:, 2:3]

        # Centered coordinates
       half_Lx = 0.5 * (xmax - xmin)  
       half_Ly = 0.5 * (ymax - ymin)  
       xc = x[:, 0:1] - (xmin + half_Lx) 
       yc = x[:, 1:2] - (ymin + half_Ly)  

        #  ux, u = (x_c^2 - 0.25) * ux_hat
        # Enforces ux = 0 at left (x=0) and right (x=1)
       ux = (xc**2 - half_Lx**2) * ux_hat

        # uy, v = (y_c^2 - 0.25) * uy_hat + (y_c + 0.5)*Δv
       uy = (yc**2 - half_Ly**2) * uy_hat + (yc + half_Ly) * v_delta

        #  Phase-field: natural BC only ---
       phi = phi_hat
        ##Alternative choices that could be used potentially for other cases
    #   phi = 0.5*(torch.tanh(phi_hat) + 1.0)
    #   phi= torch.sigmoid(phi_hat)

# Enforces uy = 0 at bottom (y=0),  uy = Δv at top (y=1)

       return ux, uy, phi

    def forward(self, x, v_delta):
           out_raw = self.forward_raw(x)
           return self.apply_BC2(x, v_delta, out_raw)

In [ ]:
def uniform_sample(x_all, w_all, n_batch):
    """
    Uniform random subsampling of the graded grid:
    Returns (x_batch, w_batch, idx) where w_batch is corrected therefore sum(w_batch * psi) is an unbiased estimator
    of the full integral, i.e. sum(w_all * psi).
    """
    N = x_all.shape[0]
    idx = torch.randint(0, N, (n_batch,), device=x_all.device)
    x_batch = x_all[idx]
    w_batch = w_all[idx] * (N / n_batch)
    return x_batch, w_batch, idx

In [ ]:
def energy_fn(model, x, wg_t, H_prev, H_init_local, v_delta):
    """
    Computes the total potential energy of the phase-field fracture problem
    at the given integration points, for the subsequent use as the DEM training loss.

    The energy is the sum, over all points, of the degraded elastic energy (psi_elastic)
    and the regularized fracture energy (psi_fracture), weighted by the quadrature
    weights wg_t. History field enforces irreversibility, drives crack growth and preserves
    the seeded crack. 
    use_split if True: spectral (tension-compression)
    **split degrades only the tensile part of the elastic energy.

    The function returns:
    E_tot: scalar total potential energy (it is the loss to be minmized)
    H_new: updated (detached) history field for the next load step
    """
    x = x.requires_grad_(True)
    ux, uy, phi = model(x, v_delta)
    u = torch.cat([ux, uy], dim=1)
    
    eps_xx, eps_yy, eps_xy = strain_calc(u, x)
    
    # Elastic energy density 
    if use_split:
        Psi_pos, Psi_neg = el_energy_spectral_split(eps_xx, eps_yy, eps_xy, lam, mu)
        Psi_drv = Psi_pos
    else:
        Psi_drv = el_energy(eps_xx, eps_yy, eps_xy, lam, mu)
        Psi_neg = torch.zeros_like(Psi_drv)
    
    # Irreversible history field
    H_new = torch.maximum(torch.maximum(Psi_drv, H_prev), H_init_local)
    
    # Phase-field gradient
    dphix, dphiy = grad_phi(phi, x)
    nabla_phi_sq = dphix**2 + dphiy**2
    
    # Degradation function 
    kappa = 1e-7
    g = (1.0 - phi)**2 + kappa
    
    # Total energy density
    if use_split:
        psi_elastic = g * Psi_pos + Psi_neg
    else:
        psi_elastic = g * Psi_drv
    
    psi_fracture = 0.5 * Gc * (phi**2 / ls + ls * nabla_phi_sq) + g * H_new
    
    E_tot = torch.sum((psi_elastic + psi_fracture).squeeze() * wg_t)
    
    return E_tot, H_new.detach()


In [ ]:
# ============================================================
# Plotting and post-processing 
# ============================================================

xP = X_f[:,0]; yP = X_f[:,1]
phi0_plot = np.zeros_like(xP)
phi0plot_t = torch.from_numpy(phi0_plot[:,None]).to(device, dtype=torch.get_default_dtype())
phiprev = phi0plot_t.clone()

grid_plot = X_f.copy()
grid_t = torch.from_numpy(grid_plot).to(device=device, dtype=torch.get_default_dtype())

tri = mtri.Triangulation(grid_plot[:,0], grid_plot[:,1])

# plt.figure(figsize=(6,5))
# plt.tricontourf(tri, H_init_np.ravel(), levels=60, cmap='jet')
# plt.colorbar(label="H_init")
# plt.gca().set_aspect("equal")
# plt.show()

def plot_phi(phi_flat):
    plt.rcParams['font.family'] = 'serif'
    plt.figure(figsize=(6,5))
    contour = plt.tricontourf(tri, phi_flat, levels=60, cmap="jet", vmin=0, vmax=1)
    cbar = plt.colorbar(contour)
    cbar.set_label(r"$\phi$", fontsize=12, labelpad=10)
    cbar.ax.tick_params(labelsize=10) 
    plt.xlabel(r"$x$", fontsize=13, labelpad=5)
    plt.ylabel(r"$y$", fontsize=13, labelpad=5)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.gca().set_aspect("equal")
    plt.tight_layout() 
    plt.show()


## Force versus displacement diagram 

from scipy.signal import savgol_filter

N_b = 800
x_top = np.linspace(0.0, L, N_b)
y_top = np.full_like(x_top, W)
xy_top_t = torch.from_numpy(np.vstack([x_top, y_top]).T).to(device=device, dtype=torch.get_default_dtype())
fdGraph = np.zeros((len(v_deltas), 2))

def compute_reaction_force(model, xy_top_t, v_delta, lam, mu, L, N_b):
    xy_b = xy_top_t.detach().clone().requires_grad_(True)
    ux_b, uy_b, phi_b = model(xy_b, v_delta)
    u_b = torch.cat([ux_b, uy_b], dim=1)
    eps_xx_b, eps_yy_b, eps_xy_b = strain_calc(u_b, xy_b)
    g_b = (1.0 - phi_b)**2 + 1e-7
    sigma_yy_b = g_b * (lam * (eps_xx_b + eps_yy_b) + 2 * mu * eps_yy_b)
    
    # Trapezoidal integration along x
    dx = L / (N_b - 1)
    w = torch.full((N_b, 1), dx, dtype=sigma_yy_b.dtype, device=sigma_yy_b.device)
    w[0] *= 0.5
    w[-1] *= 0.5
    force = -torch.sum(sigma_yy_b * w)
    
    return force

def plot_force_disp(fd_graph):
    v = fd_graph[:, 0]
    F = fd_graph[:, 1]

    window_length = 7      
    polyorder = 3     
    if len(F) > window_length:
        F_smooth = savgol_filter(F, window_length=window_length, polyorder=polyorder)
    else:
        F_smooth = F 
    plt.figure(figsize=(8, 6))
    plt.plot(v, F_smooth, linestyle='-', color='b')
    plt.xlabel('Displacement (u)')
    plt.ylabel('Reaction Force (F)')
    plt.title('Force-Displacement Diagram')
    plt.legend()
    plt.grid(True)
    plt.show()
    return v, F_smooth


def plot_loss(loss_history, step=0):
    hist = loss_history[step]
    adam_iters  = hist['adam_iters']
    adam_losses = hist['adam_losses']

# Offset LBFGS iterations to continue after Adam
    offset = adam_iters[-1] if adam_iters else 0
    lbfgs_iters  = [offset + i for i in hist['lbfgs_iters']]
    lbfgs_losses = hist['lbfgs_losses']

    plt.figure(figsize=(10, 5))
    plt.semilogy(adam_iters,  adam_losses,  label='Adam',  color='steelblue')
    plt.semilogy(lbfgs_iters, lbfgs_losses, label='L-BFGS', color='tomato')
    plt.axvline(offset, color='gray', linestyle='--', linewidth=0.8, label='Adam → L-BFGS')
    plt.xlabel('Iteration')
    plt.ylabel('Total Energy  $\mathcal{E}$  (log scale)')
    plt.title(f'Training Loss — Load Step {step}  ($v_{{\\delta}}$ = {v_deltas[step]:.3e})')
    plt.legend()
    plt.grid(True, which='both', alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Main training loop: incremental loading with per-step
# Adam (minibatch) + L-BFGS (full-grid) optmization
# ============================================================

phi_snapshots = []
load_vals  = []
loss_history = {}
phi_prev = phi0_t.clone()

peak_force = -np.inf
peak_step = 0

step_times = {}
total_start = time.time()

## NN architecture
model = PINN2D_PF(layers=[2, 50, 50, 50, 3]).to(device).double()


# Batch fraction value can be changed here!
batch_frac = 4
n_batch = X_f.shape[0] // batch_frac

for istep, v_val in enumerate(v_deltas):

    step_start = time.time()
    v_delta = torch.tensor(v_val, dtype=torch.get_default_dtype(), device=device)
    n_epochs = n_epochs_step0 if istep == 0 else n_epochs_other  # number of ADAM epochs for 1st and subsequent loading steps
    max_iter = lbfgs_step0_iter if istep == 0 else lbfgs_other_iter # number of LBFGS epochs for 1st and subsequent loading 
    print(f"\n=== Step {istep}/{len(v_deltas)-1} | v_delta={v_val:.3e} ===")

    loss_history[istep] = {'adam_iters': [], 'adam_losses': [],
                           'lbfgs_iters': [], 'lbfgs_losses': []}

    for param in model.parameters():
        param.requires_grad = True
    model.train()

    adam_lr   = 1e-3 if istep == 0 else 1e-4    # assigning learning rate
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=adam_lr)
    lbfgs = torch.optim.LBFGS(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1.0, max_iter=max_iter,
        tolerance_grad=1e-9, tolerance_change=1e-12,
        line_search_fn='strong_wolfe')

    x = xg_t.detach().clone().requires_grad_(True)
    
# ============================================================
## Stage 1: ADAM optimization on minibatched grid
# ============================================================
    adam_start = time.time()

    for epoch in range(1, n_epochs + 1):
    # minibatch sampling
        x_batch, w_batch, idx = uniform_sample(xg_t, wg_t, n_batch)
        x_batch = x_batch.detach().clone().requires_grad_(True)

        optimizer.zero_grad()
        E_tot, H_new = energy_fn(model, x_batch, w_batch, H[idx], H_init[idx], v_delta)  
        E_tot.backward()
        optimizer.step()
        loss_history[istep]['adam_iters'].append(epoch)
        loss_history[istep]['adam_losses'].append(E_tot.item())
        if epoch % print_every == 0:
            with torch.no_grad():
                _, _, phi_dbg = model(xg_t, v_delta)
            print(f"  Adam {epoch:5d} | E={E_tot.item():.4e}"
                  f" | avg(phi)={phi_dbg.mean().item():.3e}"
                  f" | t={time.time()-adam_start:.1f}s")

# ============================================================
## Stage 2: LBFGS optimization on full grid
# ============================================================

    lbfgs_iter_count = [0]
    lbfgs_start      = time.time()

    def closure():
        lbfgs.zero_grad()
        E, _ = energy_fn(model, x, wg_t, H, H_init, v_delta)
        E.backward()
        if lbfgs_iter_count[0] % print_every == 0:
            with torch.no_grad():
                _, _, phi_lbfgs = model(xg_t, v_delta)
            print(f"  LBFGS {lbfgs_iter_count[0]:5d} | E={E.item():.4e}"
                  f" | avg(phi)={phi_lbfgs.mean().item():.3e}"
                  f" | t={time.time()-lbfgs_start:.1f}s")
        lbfgs_iter_count[0] += 1
        loss_history[istep]['lbfgs_iters'].append(lbfgs_iter_count[0])
        loss_history[istep]['lbfgs_losses'].append(E.item())
        return E

    lbfgs.step(closure)

    # ============================================================
    ## History-field update and irreversibility
    # ============================================================
    x_final = xg_t.detach().clone().requires_grad_(True)
    _, H_new = energy_fn(model, x_final, wg_t, H, H_init, v_delta)
    H = H_new.detach()

    with torch.no_grad():
        _, _, phi_raw_n = model(grid_t, v_delta)
        phi_n    = torch.maximum(phi_raw_n, phi_prev)
        phi_prev = phi_n.clone()

    # ============================================================
    ## Post-processing: snapshots, reaction force, bookkeeping
    # ============================================================

    phi_snapshots.append(phi_n.detach().cpu().numpy().ravel())
    load_vals.append(v_val)

    if istep % plot_every == 0:
       plot_phi(phi_snapshots[-1])

        
    # Reaction force
    force = compute_reaction_force(model, xy_top_t, v_delta, lam, mu, L, N_b)
    force_value = -force.item()
    fdGraph[istep, 0] = v_val      
    fdGraph[istep, 1] = force_value

    
    if force_value > peak_force:
        peak_force = force_value
        peak_step  = istep

    if istep > 0 and peak_force > 0 and force_value < 0.2 * peak_force:
        print(f"\nFull fracture at step {istep} (F={force_value:.2f}, peak {peak_force:.2f}). Stopping.")
        fdGraph = fdGraph[:istep+1]
        break

    step_times[istep] = time.time() - step_start
    print(f"  Step {istep} time: {step_times[istep]:.1f}s")

fd = fdGraph[:len(load_vals)]

dis, f_smooth = plot_force_disp(fdGraph)

df = pd.DataFrame({'displacement_mm': dis, 
    'force_N': f_smooth})

#df = pd.DataFrame(fdGraph, columns=['displacement_mm', 'force_N'])
# Outputing the FD data to Kaggle (change the working directory accordingly)
df.to_excel('/kaggle/working/fd.xlsx', index=False)
print("saved")

plot_force_disp(fdGraph)
plot_loss(loss_history, step=0)
print(f"\nTotal training time: {time.time() - total_start:.1f}s")